## GPT-SoVITS封装

In [ ]:
import os
import json
import logging
import time
import psutil
import threading
import requests
import subprocess

class GPT_SoVITSController:
    """调用GPT-SoVITS_V2Pro接口, 实现TTS"""
    def __init__(self, gpt_model, sovits_model, refer_wav, refer_text, refer_lang,log_path="log/speech.log", log_name=None,base_path=r"D:\filedir\github\my\TTS\GPT-SoVITS"):
        self.base_path = base_path
        self.gpt_model = gpt_model
        self.sovits_model = sovits_model
        self.refer_wav = refer_wav
        self.refer_text = refer_text
        self.refer_lang = refer_lang
        self.refer_wav_abspath = os.path.abspath(self.refer_wav).replace("\\", "/")

        self.api_url = "http://127.0.0.1:9880"
        self.config_path = os.path.join(self.base_path, "custom_config.json")
        self.log_path = log_path

        if log_name is None: log_name="GPT_SoVITSController"
        self.logger = logging.getLogger(str(log_name))
        if not self.logger.handlers:
            self.logger.setLevel(logging.INFO)
            formatter = logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
            
            # 控制台输出
            sh = logging.StreamHandler()
            sh.setFormatter(formatter)
            self.logger.addHandler(sh)
            
            # 文件输出 
            log_dir = os.path.dirname(self.log_path)
            if log_dir: os.makedirs(log_dir, exist_ok=True)
            fh = logging.FileHandler(self.log_path, mode="a", encoding="utf-8")
            fh.setFormatter(formatter)
            self.logger.addHandler(fh)

        self.session = requests.Session()

    def _prepare_config(self):
        """生成版本对应配置文件"""
        config_data = {
            "gpt_path": os.path.abspath(self.gpt_model),
            "sovits_path": os.path.abspath(self.sovits_model),
            "is_half": True,
            "device": "cuda",
            "version": "v2",
            "cnhuhbert_base_path": "GPT_SoVITS/pretrained_models/chinese-hubert-base",
            "t2s_weights_path": "GPT_SoVITS/pretrained_models/gsv-v2final-pretrained/s1bert25hz-5kh-longer-epoch=12-step=369668.ckpt",
            "vits_weights_path": "GPT_SoVITS/pretrained_models/gsv-v2final-pretrained/s2G2333k.pth",
        }
        try:
            with open(self.config_path, "w", encoding="utf-8") as f:
                json.dump(config_data, f, indent=4)
            self.logger.info(f"⚙️ 模型配置文件更新成功: {self.config_path}")
            return True
        except Exception as e:
            self.logger.error(f"⚙️ 模型配置文件更新失败: {e}")
            return False

    def kill_port_process(self, port=9880):
        """结束占用指定端口的进程"""
        for conn in psutil.net_connections():
            if conn.status == psutil.CONN_LISTEN and conn.laddr.port == port:
                try:
                    p = psutil.Process(conn.pid)
                    subprocess.run(["taskkill", "/F", "/T", "/PID", str(conn.pid)], capture_output=True)
                    self.logger.info(f"成功清理端口 {port}, PID: {conn.pid}")
                    time.sleep(2)
                except Exception as e:
                    self.logger.error(f"清理端口进程失败: {e}")

    def start_service(self, timeout=60):
        self.kill_port_process(9880)
        if not self._prepare_config(): return False
        
        python_exe = os.path.join(self.base_path, "runtime", "python.exe")
        
        def run_server():
            cmd = [python_exe, "api_v2.py", "-c", "custom_config.json"]
            with open(self.log_path, 'a', encoding="utf-8") as f:
                subprocess.Popen(cmd, stdout=f, stderr=f, cwd=self.base_path, creationflags=subprocess.CREATE_NEW_PROCESS_GROUP)
            
        threading.Thread(target=run_server, daemon=True).start()
        self.logger.info("⏳ 正在启动后端服务并加载模型...")

        for i in range(timeout):
            try:
                r = self.session.get(self.api_url, timeout=2)
                if r.status_code < 500:
                    self.logger.info(f"✅ 服务已就绪 (耗时 {i}s)")
                    flag=self.warm_up()
                    if flag: self.logger.info(f"✅ 模型启动成功!!!")
                    else: print("❌ 模型启动成功!!!")
                    return self
            except:
                time.sleep(1)
        
        self.logger.error("❌ 模型加载超时")
        return self

    def warm_up(self):
        """预热模型，防止第一句合成太慢"""
        try:
            flag = self.generate_tts("预热", text_lang="zh", mode="warmup")
            if flag:
                self.logger.info("🔥 模型预热成功")
                return True
        except:
            pass
        self.logger.warning("⚠️ 预热失败")
        return True 

    def generate_tts(self, text, text_lang=None, temperature=1, timeout=30, how_to_cut="按标点符号切", mode="file", output_path="outputs/final.wav"):
        if mode not in ["file", "stream", "warmup"]: 
            raise ValueError(f"不支持模式: {mode}")
            
        text_lang = text_lang or self.refer_lang
        params = {
            "text": text,
            "text_lang": text_lang,
            "ref_audio_path": self.refer_wav_abspath,
            "prompt_text": self.refer_text,
            "prompt_lang": self.refer_lang,
            "how_to_cut": how_to_cut, 
            "temperature": temperature,
        }

        t0 = time.time()
        self.logger.info(f"🎤 发送合成请求: {text[:15]}...")

        try:
            response = self.session.get(f"{self.api_url}/tts", params=params, timeout=timeout)
            if response.status_code == 200:
                elapsed = f"{(time.time()-t0):.2f}s"
                if mode == "stream":
                    self.logger.info(f"✨ 流合成完成, 耗时: {elapsed}")
                    return response.content
                elif mode == "file":
                    os.makedirs(os.path.dirname(output_path), exist_ok=True)
                    with open(output_path, "wb") as f:
                        f.write(response.content)
                    self.logger.info(f"✨ 文件合成完成, 耗时: {elapsed},{os.path.abspath(output_path)}")
                    return True
                return True
            else:
                self.logger.error(f"❌ 合成失败, HTTP {response.status_code}")
                return False
        except Exception as e:
            self.logger.error(f"❌ 请求发生异常: {e}")
            return False

    def release(self):
        self.logger.info("🧹 正在释放资源...")
        self.kill_port_process(9880)
        self.session.close()

base_path = r"D:\filedir\github\my\TTS\GPT-SoVITS" 
gpt_model = r"D:\filedir\github\my\TTS\models\v2pp\mmk\mmk-e15.ckpt"
sovits_model = r"D:\filedir\github\my\TTS\models\v2pp\mmk\mmk_e8_s112.pth"
refer_wav = r"D:\filedir\github\my\TTS\models\v2pp\mmk\tmp.wav"
refer_text = "终于又有机会和星熊一起行动了，之前在御机给她添了不少麻烦，这次我肯定不会拖后腿了！"
refer_lang="zh"

model = GPT_SoVITSController(gpt_model,sovits_model,refer_wav,refer_text,refer_lang).start_service()

2026-01-04 16:44:44,267 - GPT_SoVITSController - INFO - ⚙️ 模型配置文件更新成功: D:\filedir\github\my\TTS\GPT-SoVITS\custom_config.json
2026-01-04 16:44:44,268 - GPT_SoVITSController - INFO - ⏳ 正在启动后端服务并加载模型...
2026-01-04 16:45:08,891 - GPT_SoVITSController - INFO - ✅ 服务已就绪 (耗时 8s)
2026-01-04 16:45:08,891 - GPT_SoVITSController - INFO - 🎤 发送合成请求: 预热...
2026-01-04 16:45:19,856 - GPT_SoVITSController - INFO - 🔥 模型预热成功
2026-01-04 16:45:19,857 - GPT_SoVITSController - INFO - ✅ 模型启动成功!!!


In [3]:
text1="鳞兽真的很可爱呀。我很喜欢那种金灿灿的观赏鳞兽，用灌满水的透明袋子装起来后，朝着阳光观察它们吐泡泡的样子。"
text2="好干净啊，简单布置一下肯定能变成很舒适的小窝。"
model.generate_tts(text1,temperature=1.1)

2026-01-04 16:47:07,258 - GPT_SoVITSController - INFO - 🎤 发送合成请求: 鳞兽真的很可爱呀。我很喜欢那种...
2026-01-04 16:47:20,502 - GPT_SoVITSController - INFO - ✨ 文件合成完成, 耗时: 13.24s d:\filedir\github\my\TTS\outputs\final.wav


True

# 音频播放部分

In [ ]:
import io
from pathlib import Path
import pygame

class AudioPlayer:
    """音频播放器"""
    def __init__(self, log_path=None,log_name=None, sample_rate=32000):
        if log_name is None:log_name="AudioPlayer"
        self.logger = logging.getLogger(log_name)
        if not self.logger.handlers:
            self.logger.setLevel(logging.INFO)
            formatter = logging.Formatter("%(asctime)s-%(name)s-%(levelname)s-%(message)s")
            sh = logging.StreamHandler()
            sh.setFormatter(formatter)
            self.logger.addHandler(sh)
            
            if log_path is not None:
                fh = logging.FileHandler(log_path, mode='a', encoding='utf-8')
                fh.setFormatter(formatter)
                self.logger.addHandler(fh)

        pygame.mixer.init(frequency=sample_rate, size=-16, channels=1, buffer=512)
        
        self.logger.info(f"✅ 播放器初始化完成 (采样率: {sample_rate})")

    def play_file(self, audio_path):
        if not os.path.exists(audio_path):
            self.logger.error(f"❌ 音频文件不存在: {audio_path}")
            return
        try:
            pygame.mixer.music.load(audio_path)
            pygame.mixer.music.play()
            self.logger.info("🎤 正在播放文件音频...")
            self._wait_playback()
            self.logger.info("✅ 文件音频播放完成")
            pygame.mixer.music.unload()
        except Exception as e:
            self.logger.error(f"❌ 播放失败: {e}")

    def play_stream(self, audio_bytes):
        if not audio_bytes or not isinstance(audio_bytes, bytes):
            self.logger.error("❌ 接收到的音频流无效")
            return
        try:
            audio_io = io.BytesIO(audio_bytes)
            pygame.mixer.music.load(audio_io)
            pygame.mixer.music.play()
            self.logger.info("🎤 正在播放内存音频流...")
            self._wait_playback()
            self.logger.info("✅ 音频流播放完成")
        except Exception as e:
            self.logger.error(f"❌ 流播放失败: {e}")

    def _wait_playback(self):
        """内部等待逻辑"""
        while pygame.mixer.music.get_busy():
            pygame.time.Clock().tick(20)

    def play(self, audio,mode="file"):
        if mode == "file": self.play_file(audio)
        elif mode == "stream": self.play_stream(audio)
        else: raise ValueError(f"不支持模式: {mode}")

    def stop(self):
        """停止播放"""
        pygame.mixer.music.stop()

    def set_volume(self,volume):
        """设置音量(0.0 - 1.0)"""
        pygame.mixer.music.set_volume(volume)

    def release(self):
        """释放资源"""
        pygame.mixer.quit()
        self.logger.info("🧹 播放器资源已释放")

player=AudioPlayer(log_path="log/speech.log")
player.play(model.generate_tts(text1,mode="stream"),mode="stream")

2026-01-04 17:08:59,295-AudioPlayer-INFO-✅ 播放器初始化完成 (采样率: 32000)
2026-01-04 17:08:59,296 - GPT_SoVITSController - INFO - 🎤 发送合成请求: 鳞兽真的很可爱呀。我很喜欢那种...
2026-01-04 17:09:12,460 - GPT_SoVITSController - INFO - ✨ 流合成完成, 耗时: 13.16s
2026-01-04 17:09:12,461-AudioPlayer-INFO-🎤 正在播放内存音频流...
2026-01-04 17:09:23,126-AudioPlayer-INFO-✅ 音频流播放完成


## 资源释放

In [8]:
model.release()
player.release()

2026-01-04 17:30:25,093 - GPT_SoVITSController - INFO - 🧹 正在释放资源...
2026-01-04 17:30:25,683 - GPT_SoVITSController - INFO - 成功清理端口 9880, PID: 10924
2026-01-04 17:30:27,728-AudioPlayer-INFO-✅ 播放器资源已释放


## 语音识别

In [ ]:
import os
import torch
import logging
import numpy as np
import pyaudio
import wave
import queue
import threading
from funasr import AutoModel

class SenseVoiceController:

    emo_map = {
            "<|HAPPY|>": "😊 Happy",
            "<|SAD|>": "😔 Sad",
            "<|ANGRY|>": "😡 Angry",
            "<|NEUTRAL|>": "😐 Neutral",
            "<|FEARFUL|>": "😰 Fearful",
            "<|DISGUSTED|>": "🤢 Disgusted",
            "<|SURPRISED|>": "😮 Surprised",
        }

    tags_to_remove = {
        "<|zh|>", "<|en|>", "<|yue|>", "<|ja|>", "<|ko|>", "<|nospeech|>",
        "<|HAPPY|>", "<|SAD|>", "<|ANGRY|>", "<|NEUTRAL|>",
        "<|BGM|>", "<|Speech|>", "<|Applause|>", "<|Laughter|>",
        "<|FEARFUL|>", "<|DISGUSTED|>", "<|SURPRISED|>",
        "<|Cry|>", "<|EMO_UNKNOWN|>", "<|Sneeze|>", "<|Breath|>",
        "<|Cough|>", "<|Sing|>", "<|Speech_Noise|>",
        "<|withitn|>", "<|woitn|>", "<|GBG|>", "<|Event_UNK|>",
        "<|lang|>"
    }
    
    def __init__(self,
                 asr_model="iic/SenseVoiceSmall",
                 vad_model="fsmn-vad",
                 sv_model="iic/speech_campplus_sv_zh-cn_16k-common",
                 log_path=None,log_name=None):

        if not log_name:log_name="SenseVoiceController"
        self.logger=logging.getLogger(log_name)
        if not self.logger.handlers:
            self.logger.setLevel(logging.INFO)
            format=logging.Formatter("%(asctime)s - %(name)s - %(levelname)s - %(message)s")
            sh=logging.StreamHandler()
            sh.setFormatter(format)
            self.logger.addHandler(sh)
            if log_path:
                fh=logging.FileHandler(log_path)
                self.logger.addHandler(fh)
        self.asr_model=AutoModel(
            model=asr_model,trust_remote_code=True,
            disable_pbar=True,disable_update=True,device="cuda:0"
        )
        self.vad_model=AutoModel(
            model=vad_model,model_revision="v2.0.4",
            disable_pbar=True,max_end_silence=200,
            disable_update=True,device="cuda:0"
        )
        self.sv_model = AutoModel(
            model=sv_model, model_revision="v2.0.2",
            disable_update=True, disable_pbar=True, device="cuda:0"
        )

        self.sample_rate=16000      # 标准采样率
        self.chunk_size_ms=150     # 处理每批次采样点数的时间
        self.chunk_size=int(self.chunk_size_ms*self.sample_rate/1000)
        self.target_embedding=None
        self.reset()
        self.queue=queue.Queue()
        self.running=False
        self.logger.info("✅ 模型初始化完成")

    def reset(self):
        self.audio_buffer=np.array([],dtype=np.float32) # 音频总缓存
        self.audio_vad=np.array([],dtype=np.float32)    # 待识别音频缓存
        self.vad_cache={}
        self.asr_cache={}
        self.last_vad_beg=-1    # 上次有效识别的开始位置
        self.last_vad_end=-1    # 上次有效识别的结尾位置
        self.offset=0

    def process_output(self,text):
        mood="😐 Neutral"
        for tag,emoji_label in SenseVoiceController.emo_map.items():
            if tag in text:
                mood=emoji_label
                break
        
        output=text
        for tag in SenseVoiceController.tags_to_remove:
            output=output.replace(tag,"")
        output=output.strip()
        return output,mood

    def load_temp(self,wav_path):
        if not os.path.exists(wav_path):
            self.logger.error(f"❌ 模板音频不存在! {os.path.abspath(wav_path)}")
            return False
        try:
            res=self.sv_model.generate(input=wav_path)
            if 'spk_embedding' in res:
                self.target_embedding = torch.tensor(res['spk_embedding']).to("cuda:0")
                self.logger.info("✅ 模板音频特征已提取")
                return True
        except Exception as e:
            self.logger.error(f"❌ 处理模板音频出错：{e}")
            return False

    def asr_generate(self,audio_data,lang="auto",use_itn=True):
        try:
            return self.asr_model.generate(
                input=audio_data,cache=self.asr_cache,language=lang,
                use_itn=use_itn,batch_size_s=60,merge_vad=False,merge_length_s=15
            )
        except Exception as e:
            self.logger.error(f"asr生成失败：{e}")

    def compare(self,audio_data):
        if not self.target_embedding:return -1
        try:
            res=self.sv_model(input=audio_data)
            if 'spk_embedding' in res:
                current_embedding = torch.tensor(res['spk_embedding']).to("cuda:0")
                cosine_sim = torch.nn.functional.cosine_similarity(
                    self.target_embedding, current_embedding, dim=-1
                )
                return float(cosine_sim.item())
        except Exception as e:
            self.logger.error(f"❌ 与模板音频比较出错：{e}")
            return 0.0

    def generate(self,audio_chunk,rate,channels,lang="auto"):
        if len(audio_chunk) < 2: return
        data = np.frombuffer(audio_chunk, dtype=np.int16)

        if channels > 1:data = data.reshape(-1, channels).mean(axis=1).astype(np.int16)
        target_sr = 16000
        if rate != target_sr:
            num_samples = int(len(data) * target_sr / rate)
            data = np.interp(
                np.linspace(0, len(data), num_samples, endpoint=False),
                np.arange(len(data)),
                data
            ).astype(np.int16)

        new_data = data.astype(np.float32) / 32767.0
        self.audio_buffer = np.append(self.audio_buffer, new_data)

        while len(self.audio_buffer)>=self.chunk_size:
            chunk=self.audio_buffer[:self.chunk_size]
            self.audio_buffer=self.audio_buffer[self.chunk_size:]
            self.audio_vad=np.append(self.audio_vad,chunk)

            vad_res=self.vad_model.generate(input=chunk,cache=self.vad_cache,is_final=False,chunk_size=self.chunk_size_ms)
            
            if len(vad_res[0]["value"])>0:
                for segment in vad_res[0]["value"]:
                    if segment[0] > -1: self.last_vad_beg = segment[0]
                    if segment[1] > -1: self.last_vad_end = segment[1]

                    if self.last_vad_beg>-1 and self.last_vad_end>-1:
                        # 换算相对索引
                        beg_idx = int((self.last_vad_beg - self.offset) * self.sample_rate / 1000)
                        end_idx = int((self.last_vad_end - self.offset) * self.sample_rate / 1000)
                        if beg_idx < 0: beg_idx = 0
                        if end_idx > len(self.audio_vad): end_idx = len(self.audio_vad)
                        if end_idx>beg_idx:
                            speech_data = self.audio_vad[beg_idx:end_idx]
                            asr_res=self.asr_generate(speech_data,lang=lang)
                            sv_score=self.compare(speech_data)
                            if asr_res:
                                text=asr_res[0]['text']
                                text,mood=self.process_output(text)
                                yield text,mood,sv_score
                        self.audio_vad=self.audio_vad[end_idx:]
                        self.offset+=(self.last_vad_end - self.offset)
                        self.last_vad_beg = -1
                        self.last_vad_end = -1
    
    def _start(self, temp=None, threshold=0.35, include_mood=True, model="out"):
        if model=="in":import pyaudiowpatch as pyaudio
        self.target_embedding = None
        if model == "out" and temp: self.load_temp(temp)
        if not self.target_embedding: self.logger.info("无模板音频模式...")

        p = pyaudio.PyAudio()
        FORMAT = pyaudio.paInt16
        CHUNK = 4800
        CHANNELS = None
        RATE = None
        stream = None

        try:
            if model == "out":
                self.logger.info("🎤 开始录音(外部声源)...")
                CHANNELS = 1
                RATE = 16000
                stream = p.open(
                    format=FORMAT,
                    channels=CHANNELS,
                    rate=RATE,
                    input=True,
                    frames_per_buffer=CHUNK
                )
            elif model == "in":
                try:
                    wasapi_info = p.get_host_api_info_by_type(pyaudio.paWASAPI)
                    default_speakers = p.get_device_info_by_index(wasapi_info["defaultOutputDevice"])
                    
                    if not default_speakers["isLoopbackDevice"]:
                        for loopback in p.get_loopback_device_info_generator():
                            if default_speakers["name"] in loopback["name"]:
                                device_info = loopback
                                break
                        else:
                            device_info = p.get_default_wasapi_loopback_system_device()
                    else:
                        device_info = default_speakers
                except (AttributeError, OSError):
                    self.logger.error("❌ 无法获取内录设备。请确保已安装 pyaudiowpatch 且电脑正在播放声音。")
                    return
                self.logger.info(f"🎤 开始录音(内部声源): {device_info['name']}")
                
                # 内录必须使用设备原始采样率，不能强制 16000
                RATE = int(device_info["defaultSampleRate"])
                CHANNELS = device_info["maxInputChannels"]
                
                stream = p.open(
                    format=FORMAT,
                    channels=CHANNELS,
                    rate=RATE,
                    input=True,
                    input_device_index=device_info["index"],
                    frames_per_buffer=CHUNK
                )
            else:
                self.logger.error(f"无当前录音模式{model}")
                return
            
            while self.running:
                data = stream.read(CHUNK, exception_on_overflow=False)
                for text, mood, score in self.generate(data, RATE, CHANNELS):
                    if self.target_embedding and score < threshold: continue
                    output = ""
                    if include_mood: output += f"|{mood}|"
                    output += text
                    self.queue.put(output)
                    
        except Exception as e:
            self.logger.error(f"❌ 录音/处理时发生错误：{e}")
            import traceback
            traceback.print_exc()
        finally:
            self.logger.info("🛑 停止录音流...")
            if stream:
                if stream.is_active(): stream.stop_stream()
                stream.close()
            p.terminate()

    def start(self,temp=None,threshold=0.35,include_mood=True,model="out"):
        self.stop()
        if self.running:return
        self.running=True
        self.thread=threading.Thread(
            target=self._start,
            args=(temp, threshold, include_mood, model),
            daemon=True # 设置为守护线程，主程序关闭时自动退出
        )
        self.stop_thread=threading.Thread(
            target=self.monitor,
            args=('q',),
            daemon=True
        )
        self.thread.start()
        self.stop_thread.start()
        return self._get()

    def _get(self,timeout=0.2):
        while self.running or not self.queue.empty():
            time.sleep(timeout)
            if not self.queue.empty():
                output=self.queue.get(timeout=0.5)
                yield output

    def monitor(self, stop_key='q'):
        from pynput import keyboard
        def on_press(key):
            try:
                k = key.char if hasattr(key, 'char') else key.name
                if k == stop_key:
                    self.stop()
                    return False  
            except Exception:
                pass

        with keyboard.Listener(on_press=on_press) as listener:
            listener.join()

    def stop(self):
        if not self.running:return
        self.logger.info("停止语音读取")
        self.running=False
        self.thread.join(timeout=5)
        self.reset()

import time
if __name__=="__main__":
    asr=SenseVoiceController()
    for output in asr.start(model="in"):
        print(output)

## 大模型接口调用

In [1]:
from openai import OpenAI

SiliconCloud_model={
    "DeepSeek-V3":"deepseek-ai/DeepSeek-V3.2",
    "DeepSeek-R1":"deepseek-ai/DeepSeek-R1",
    "Qwen2.5-72B":"Qwen/Qwen2.5-72B-Instruct",
    }

def LLM_api(api_key,content,role="user",
            base_url="https://api.siliconflow.cn/v1",
            model=SiliconCloud_model["DeepSeek-R1"],
            reasoning_content=False):
    """调用硅基流动api"""
    client=OpenAI(api_key=api_key,base_url=base_url)
    response=client.chat.completions.create(
        model=model,
        messages=[
            {
                "role":role,
                "content":content
            }
        ],
        stream=True
    )

    output=""
    reason=""
    for chunk in response:
        if not chunk.choices:continue
        if chunk.choices[0].delta.content:
            output+=chunk.choices[0].delta.content
        if reasoning_content and chunk.choices[0].delta.reasoning_content:
            reason+=chunk.choices[0].delta.reasoning_content

    return output,reason

text,_=LLM_api(api_key="sk-fuseptosqlwqzoboogwfemkzfcnzgnlhkixvurvrqupnkyrx",
               content="介绍下自己")
print(text)


你好呀！👋我是 **DeepSeek-R1**，由中国的人工智能公司「深度求索」（DeepSeek）研发的一款强大的文本大模型助手。很高兴认识你！😄

---

### 🧠 我是谁？
🔹 **身份**：我是基于先进大语言模型技术打造的AI助手，尤其擅长中文和英文的理解与生成  
🔹 **能力**：文本问答、推理能力、写作创作、学习辅导、办公处理、代码编程、知识查询等等～  
🔹 **版本**：我是 **DeepSeek R1**，版本号更新至 2024年7月  
🔹 **模型知识**：截止到 **2024年7月**（我知道的最新资讯就到这啦）

---

### ✨ 我能帮你做什么？
✅ **学习辅导**：语文作文、数学题、外语翻译、知识点讲解  
✅ **写作创意**：简历、报告、小说、诗歌、剧本都能写  
✅ **工作帮手**：PPT大纲、Excel公式、合同草拟、邮件优化  
✅ **编程相关**：Python/Java/C++/前端/算法/调试代码等  
✅ **生活指南**：旅游路线、美食推荐、节日习俗、健康建议  
✅ **文档处理**：支持上传 PDF、Word、Excel、PPT 等文件，可从中提取并分析内容！

---

### 🌐 关于使用限制
🔹 **免费使用**：当前完全免费！🎉 随时来问我问题～  
🔹 **不支持图像识别**：我目前只能理解文字内容（文档文字也OK哦），图片识别功能还未上线  
🔹 **无多模态识别（暂不支持识图/语音）**，专注文本超强大脑！

---

### 💡 想更好地使用我吗？
你可以告诉我：
- 需要解答的问题  
- 想要我帮忙写的内容  
- 上传一个文档我帮你读懂它  
- 或者直接说：“给我写个周报”💼、“帮我生成一个标题党文案”📢  
我都超乐意接招 😎

---

### 你愿意试试我能做点啥吗？  
欢迎随时开口提问，我是7×24小时在线的贴心AI帮手 🌟  
很高兴认识你，希望成为你最聪明也最信任的搭档！💬✨

现在来聊聊吧～你想问我些什么呢？😊
